# Chapter 12 &mdash; Acceptance by Final State or Empty Stack; Deterministic PDA

**Concept 5 of the Chapter 12 decomposition:** *Acceptance by Final State or Empty Stack; Deterministic PDA*

IDs, computations, two acceptance policies &mdash; and only nondeterministic PDA capture all CFLs.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Acceptance-Policies/Concept-Acceptance-Policies.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


An **instantaneous description** (ID) is a triple *(state, remaining input, stack)*. A
**computation** is a chain of IDs linked by $\Delta$.

There are **two acceptance policies**:

* **`ACCEPT_F`** &mdash; the input is exhausted in a final state;
* **`ACCEPT_S`** &mdash; the input is exhausted with an **empty stack**.

They recognise the **same class** of languages, and either can be converted to the
other, but a *particular* machine may accept different languages under the two
policies.

A PDA is **deterministic** if no ID ever has two moves &mdash; counting $\varepsilon$
moves, which is the part beginners forget. DPDA are strictly weaker (Chapter 11,
Concept 17).

## 2. Definitions

### A machine, run under both policies

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
Dyck = md2mc('''PDA
!! Push on '(', pop on ')', accept when the input is gone and # is on top.
I : ( , #  ; (#  -> I     !! first '(' -- push it above the bottom marker
I : ( , (  ; ((  -> I     !! another '(' -- push
I : ) , (  ; ''  -> I     !! ')' matches -- pop
I : '' , # ; #   -> F     !! nothing left and stack is just # -- accept
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### A determinism check

In [ ]:
def deterministic(P):
    bad = []
    for (q, inp, pop), outs in P["Delta"].items():
        if len(outs) > 1:
            bad.append(((q, inp, pop), 'two moves on one label'))
        if inp != '' and (q, '', pop) in P["Delta"]:
            bad.append(((q, inp, pop), 'conflicts with an epsilon move'))
    return not bad, bad

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;4.&nbsp;The Formal PDA $(Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Formal-PDA/Concept-Formal-PDA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch12&nbsp;6.&nbsp;Exploring PDA in Jove: Reading ID Traces](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Exploring-PDA-In-Jove/Concept-Exploring-PDA-In-Jove.ipynb)&nbsp;&rarr;

---

## 3. Tests

An ID is (state, remaining input, stack).

In [ ]:
surv, paths, visited = run_pda('(())', Dyck, STKMAX=8)
print("the accepting computation, as a chain of IDs:")
for idd in paths[0][1]:
    print("   ", idd)
print("\nfinal ID :", paths[0][0])

The stack shrinks back to `#`, which is what `ACCEPT_F` then needs.

In [ ]:
chain = paths[0][1]
stacks = [st for (_, _, st) in chain]
print("stack at each step :", stacks)
assert stacks[0] == '#' and stacks[-1] == '#'

**The two policies differ for this machine.**

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(7) for p in product('()', repeat=k)]
f = {s for s in strs if pda_accepts(Dyck, s, acceptance='ACCEPT_F', STKMAX=9)}
e = {s for s in strs if pda_accepts(Dyck, s, acceptance='ACCEPT_S', STKMAX=9)}
print("ACCEPT_F accepts %d strings" % len(f))
print("ACCEPT_S accepts %d strings" % len(e))
print("only in F :", sorted(f - e)[:5])
print("only in S :", sorted(e - f)[:5])
print("\nSame machine, two policies, two languages -- the CLASSES coincide,")
print("not the individual machines.")

This machine happens to be deterministic.

In [ ]:
ok, bad = deterministic(Dyck)
print("Dyck deterministic? ", ok, bad)

A genuinely nondeterministic machine: $\{ww^R\}$ must guess the midpoint.

In [ ]:
Pal = md2mc('''PDA
I : a , # ; A#  -> I
I : a , A ; AA  -> I
I : a , B ; AB  -> I
I : b , # ; B#  -> I
I : b , A ; BA  -> I
I : b , B ; BB  -> I
I : '' , # ; #  -> M    !! GUESS: the midpoint is here
I : '' , A ; A  -> M
I : '' , B ; B  -> M
M : a , A ; ''  -> M
M : b , B ; ''  -> M
M : '' , # ; #  -> F
''')
ok, bad = deterministic(Pal)
print("Pal deterministic? ", ok)
print("conflicts :", bad[:3])
assert not ok
for s in ['', 'aa', 'abba', 'ab', 'aab']:
    print("  %-7r in ww^R? %-6s PDA %s"
          % (s, s == s[::-1] and len(s) % 2 == 0,
             pda_accepts(Pal, s, STKMAX=8)))
assert pda_accepts(Pal, 'abba', STKMAX=8)
assert not pda_accepts(Pal, 'ab', STKMAX=8)

## 4. Exercises


1. Convert the Dyck PDA to accept by empty stack. What do you add?
2. Why do $\varepsilon$ moves count against determinism?
3. Give a language accepted by a DPDA under `ACCEPT_F` but not under `ACCEPT_S`.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-Acceptance-Policies')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')